# 🧪 Semi-supervised and knowledge-guided clustering

**Xu Lab | Biology-to-CryoEM Research | Module 24**  
**Phase:** F: Research frontiers | **Level:** scaffolded undergraduate → research | **Time:** 60–100 minutes  
**Hardware:** Google Colab free-tier CPU (optional T4 GPU for later extensions)  
**Core tools:** Python, NumPy, SciPy, Matplotlib; selected modules use scikit-learn/PyTorch  
**Prerequisites:** Previous tutorial  
**Status:** self-contained *pedagogical demonstration* using synthetic data, not an official reproduction of Xu Lab publications.  
**Safety:** for research and educational purposes only, not for clinical diagnosis.

## Learning objectives

1. Explain the biological question and physical limitations.
2. Derive or interpret the model below and identify its assumptions.
3. Run the experiment, change a parameter, and interpret quantitative output.
4. Explain how the concept supports the research papers linked below.

## Biological motivation

A few trustworthy expert constraints can guide discovery without labeling every particle.

## Conceptual bridge

Combine feature space with sparse must-link/cannot-link constraints and inspect agreement; do not mistake the illustration for a published algorithm.

## Mathematical model

$\mathcal L=\mathcal L_{\mathrm{cluster}}+\lambda\mathcal L_{\mathrm{constraints}}$.

## Notebook roadmap

1. Reproducible setup  2. Synthetic experiment  3. Interpretation  4. Challenge  5. Paper connection.


In [ ]:
# -- Environment check and shared utilities --
import sys, importlib.util
import numpy as np
import scipy
from scipy import ndimage as ndi, signal
import matplotlib.pyplot as plt
SEED=2026
rng=np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi':95,'axes.grid':False})
print('Python:',sys.version.split()[0], 'NumPy:',np.__version__)
print('PyTorch available:',bool(importlib.util.find_spec('torch')))
try:
    import torch
    print('CUDA available:',torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch not installed; required only for Module 13/14')
# Colab stock environment normally contains the required libraries.
# If an import fails, run: %pip install -q numpy scipy matplotlib scikit-learn torch

def phantom(n=48):
    """Build a deterministic 3D synthetic density with three distinct macromolecular blobs."""
    zz, yy, xx = np.mgrid[:n,:n,:n].astype(float)
    rho = np.zeros((n,n,n),dtype=float)
    for z,y,x,s,amp in [(0.45*n,.47*n,.48*n,.10*n,1.),(.65*n,.38*n,.58*n,.07*n,.7),(.40*n,.66*n,.36*n,.075*n,.8)]:
        rho += amp*np.exp(-((zz-z)**2+(yy-y)**2+(xx-x)**2)/(2*s*s))
    return rho/np.max(rho)


## Experiment: build, inspect, and check a scientific claim

Run the code below, then identify (i) input, (ii) transformation, (iii) output and (iv) a key assumption.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
X,y=make_blobs(n_samples=120,centers=[[-2,0],[0,1.5],[2,0]],cluster_std=1.0,random_state=SEED)
pred=KMeans(n_clusters=3,random_state=SEED,n_init=10).fit_predict(X)
# A must-link pair constraint: same-label probability is a target, not a guarantee from KMeans.
pairs=[(0,int(np.where(y==y[0])[0][-1])),(1,int(np.where(y==y[1])[0][-1]))]
print('Pairs of expert-confirmed same-class points:',pairs)
print('Baseline ARI:',round(adjusted_rand_score(y,pred),3))
# Simple constraint injection illustration: make must-linked features identical before clustering.
# This is NOT the full knowledge-guided representation-learning paper method.
X_linked=X.copy()
for i,j in pairs:
    mean_pair=(X_linked[i]+X_linked[j])/2
    X_linked[i]=mean_pair;X_linked[j]=mean_pair
pred_linked=KMeans(n_clusters=3,random_state=SEED,n_init=10).fit_predict(X_linked)
print('Must-link heuristic ARI:',round(adjusted_rand_score(y,pred_linked),3))
for i,j in pairs:print('Pair clustered together?',bool(pred[i]==pred[j]))
plt.scatter(X[:,0],X[:,1],c=pred,s=12);plt.show()

## Explain what happened

**Expected scientific observation:** Prior constraints can improve meaningful grouping, but label errors and biased priors can also suppress novel discoveries.

Write 3–5 sentences connecting the plotted result to the biological motivation. Describe one failure case and one alternative explanation.

## Guided challenge (edit code above)

Add a penalty that pulls confirmed pairs together before K-means. Compare with labels randomly swapped to simulate incorrect expert knowledge.

**Deliverable:** a labeled figure or small results table, a claim, and one limitation. Quantify the effect rather than just saying “it looks better.”

## Research connection and reading strategy

**Related Xu Lab research:** 2023 DISCA; 2026 knowledge-guided semi-supervised clustering.

Read each paper in this order: biological question → observations/data → forward model → key objective → baseline → evaluation split → limitations. Distinguish what the toy exercise illustrates from the actual published implementation. See `../PAPER_TO_TUTORIAL_MAP.md` for publication titles and further prerequisites.

## Exit ticket

1. What is known from the measured data, versus assumed by the model?
2. Which plotted quantity could be misleading if evaluated only on synthetic data?
3. Sketch one controlled ablation and one biological validation experiment.
4. Save an updated notebook and a 150-word lab note.